# RAG on QASPER & PubMedQA — A Core-Code Walkthrough

This notebook walks through the **entire data and retrieval stack** of the project, end to end:

```
raw QASPER / PubMedQA
   -> cleaning & flattening        (scripts/01)
   -> evidence alignment           (scripts/02)
   -> chunking                     (scripts/03, src/rag_data/chunker.py)
   -> executable data contract     (src/rag_data/validate.py)
   -> dense retrieval + evaluation (baselines/member2_dense_rag)
   -> PubMedQA citation-graph expansion (scripts/20, scripts/21)
   -> graph-enhanced retrieval     (1-hop realisation of CG-RAG)
```

**Everything runs on the committed artifacts** — no downloads, no re-running the pipeline.
Runtime is roughly 10–20 seconds on a laptop CPU.

**Dependencies:** `pandas`, `pyarrow`, `numpy`, `tokenizers`.
If `sentence-transformers` + `faiss` are present, retrieval uses real **BGE** embeddings;
otherwise it automatically falls back to a **hashing bag-of-words** encoder so the notebook
still runs. The fallback measures lexical overlap only — it is a *plumbing demo*, not a
BGE result. Every cell that depends on this prints the active backend.

## 0. Environment and data inventory

The repository keeps only the artifacts downstream members actually consume.
Everything else (raw PDFs, ablation grids, generated copies) is git-ignored and reproducible.

In [1]:
import json
import os
import re
import sys
import time
import zlib
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(".").resolve()
print("working directory:", ROOT)
print("python:", sys.version.split()[0], "| pandas:", pd.__version__, "| numpy:", np.__version__)

FILES = {
    "QASPER paragraphs": "data/processed/paragraphs.parquet",
    "QASPER questions": "data/processed/qa.parquet",
    "QASPER chunks": "data/processed/chunks/size512_ov102_soft/chunks.parquet",
    "QASPER gold map": "data/processed/chunks/size512_ov102_soft/qa_with_gold.jsonl",
    "PubMedQA graph chunks": "data/processed/pubmedqa_graph/chunks.parquet",
    "PubMedQA graph questions": "data/processed/pubmedqa_graph/qa.parquet",
    "PubMedQA graph edges": "data/processed/pubmedqa_graph/graph_edges.parquet",
    "PubMedQA graph papers": "data/processed/pubmedqa_graph/papers.parquet",
    "BGE tokenizer": "data/raw/bge_tok/tokenizer.json",
}
for name, p in FILES.items():
    print(f"{name:26s} {'OK' if os.path.exists(p) else 'MISSING':7s} {p}")

try:
    import faiss  # noqa: F401
    import sentence_transformers  # noqa: F401
    BACKEND = "bge"
except Exception:
    BACKEND = "hashing"
print("\nretrieval backend:", BACKEND)

working directory: D:\Users\NTU\6405\project\RAG_Project
python: 3.13.14 | pandas: 3.1.0 | numpy: 2.5.4
QASPER paragraphs          OK      data/processed/paragraphs.parquet
QASPER questions           OK      data/processed/qa.parquet
QASPER chunks              OK      data/processed/chunks/size512_ov102_soft/chunks.parquet
QASPER gold map            OK      data/processed/chunks/size512_ov102_soft/qa_with_gold.jsonl
PubMedQA graph chunks      OK      data/processed/pubmedqa_graph/chunks.parquet
PubMedQA graph questions   OK      data/processed/pubmedqa_graph/qa.parquet
PubMedQA graph edges       OK      data/processed/pubmedqa_graph/graph_edges.parquet
PubMedQA graph papers      OK      data/processed/pubmedqa_graph/papers.parquet
BGE tokenizer              OK      data/raw/bge_tok/tokenizer.json

retrieval backend: hashing


## 1. The two datasets

They are not interchangeable — they answer different questions and demand different protocols.

| | QASPER v0.3 | PubMedQA-1k (expanded) |
|---|---|---|
| Domain | NLP conference papers | biomedical abstracts |
| Documents | 1,169 papers, full text | 1,395 papers (951 sources + 444 bridges) |
| Questions | 3,598 (2,991 with gold) | 1,000 (951 resolvable) |
| Question style | human-written, often paper-bound ("they", "this paper") | self-contained scientific claims |
| Retrieval scope | **within one paper** | **whole corpus (cross-document)** |
| Evidence | paragraph-level gold | gold = the source paper's chunks |

QASPER asks "find the evidence inside *this* paper"; PubMedQA (expanded) asks
"find the evidence *anywhere in the corpus*". That difference is what makes the
citation graph useful in one case and useless in the other.

In [2]:
pa = pd.read_parquet("data/processed/paragraphs.parquet")
qa = pd.read_parquet("data/processed/qa.parquet")
ck = pd.read_parquet("data/processed/chunks/size512_ov102_soft/chunks.parquet")
g_ck = pd.read_parquet("data/processed/pubmedqa_graph/chunks.parquet")
g_qa = pd.read_parquet("data/processed/pubmedqa_graph/qa.parquet")
g_pa = pd.read_parquet("data/processed/pubmedqa_graph/papers.parquet")
g_ed = pd.read_parquet("data/processed/pubmedqa_graph/graph_edges.parquet")

rows = [
    ("QASPER", qa.paper_id.nunique(), len(qa), len(pa), int(pa.n_tokens.sum()), len(ck)),
    ("PubMedQA graph", g_pa.paper_id.nunique(), len(g_qa), len(g_pa), int(g_pa.abstract.str.len().sum()), len(g_ck)),
]
print(f"{'dataset':<16}{'papers':>8}{'questions':>11}{'docs':>8}{'tokens/chars':>14}{'chunks':>8}")
for r in rows:
    print(f"{r[0]:<16}{r[1]:>8}{r[2]:>11}{r[3]:>8}{r[4]:>14}{r[5]:>8}")
print("\nQASPER eligible denominator (zero_gold_reason == ''):", int((qa.zero_gold_reason == "").sum()))
print("PubMedQA graph questions with gold:", int(g_qa.gold_chunk_ids.str.len().gt(0).sum()))

dataset           papers  questions    docs  tokens/chars  chunks
QASPER              1169       3598   61317       6433441   17653
PubMedQA graph      1395        951    1395       1771804    3707

QASPER eligible denominator (zero_gold_reason == ''): 2991
PubMedQA graph questions with gold: 951


## 2. The data contract: three coordinate systems

Almost every downstream bug traces back to mixing these up:

```
paper flat text  --char_start/char_end-->  chunk
   └-- paragraph text_clean --para_spans[a:b]--> the fragment inside a chunk
```

* `para_id = "{paper_id}#p{idx:04d}"` — globally stable, independent of chunking.
* `chunk_id = "{paper_id}#s{sec}#c{idx:03d}"` — stable **only within one config**.
* `para_spans` offsets are relative to **the paragraph's own text**, not the chunk.
  In parquet they are stored as JSON text because Arrow cannot infer a `(str,int,int)` tuple column.

The invariant `len(para_spans) == len(para_ids)` must hold for every chunk.

In [3]:
violations = 0
lengths_ok = 0
for r in ck.itertuples():
    spans = json.loads(r.para_spans)
    if len(spans) != len(r.para_ids):
        violations += 1
    if all(len(s) == 3 for s in spans):
        lengths_ok += 1

print("chunks checked:", len(ck))
print("para_spans/para_ids length violations:", violations)
print("well-formed (pid, start, end) triples:", lengths_ok)

sample = ck.iloc[0]
spans = json.loads(sample.para_spans)
print("\nsample chunk:", sample.chunk_id, "| section:", sample.section_name, "| tokens:", int(sample.n_tokens))
print("  paragraphs:", list(sample.para_ids))
print("  spans (relative to each paragraph):", spans[:3], "..." if len(spans) > 3 else "")
print("  char range in paper flat text:", (int(sample.char_start), int(sample.char_end)))

chunks checked: 17653
para_spans/para_ids length violations: 0
well-formed (pid, start, end) triples: 17653

sample chunk: 1503.00841#sabs#c000 | section: Abstract | tokens: 496
  paragraphs: ['1503.00841#p0000', '1503.00841#p0001', '1503.00841#p0002', '1503.00841#p0003', '1503.00841#p0004']
  spans (relative to each paragraph): [['1503.00841#p0000', 0, 606], ['1503.00841#p0001', 0, 406], ['1503.00841#p0002', 0, 677]] ...
  char range in paper flat text: (0, 2577)


## 3. Preprocessing pipeline (scripts 01 → 02)

**3.1 Profiling** (`scripts/00_profile_dataset.py`) keeps raw QASPER intact and reports sizes.

**3.2 Cleaning + flattening** (`scripts/01_prepare_corpus.py`):
QASPER stores each section as a list of paragraphs with inline placeholders (`BIBREF0`, `TABLE0`, …).
The pipeline strips the placeholders into `text_clean`, keeps `text_raw`, and concatenates all
paragraphs of a paper into one flat string with `char_start/char_end` offsets.
`offset_map` records the cleaned→raw character mapping so citations can be traced back.

**3.3 Evidence alignment** (`scripts/02_align_evidence.py`):
gold evidence arrives as *section/paragraph indices plus character-level spans*.
The script maps them onto `para_id`s, records failures, and classifies every question with a
`zero_gold_reason` (`no_evidence`, `float_only`, `align_fail`). Questions that cannot be mapped
are **excluded from the recall denominator, never scored as zero**.

In [4]:
diff = pa[pa.text_raw != pa.text_clean]
print("paragraphs whose raw text differs from cleaned text:", len(diff), f"({len(diff)/len(pa)*100:.1f}%)")
if len(diff):
    r = diff.iloc[0]
    print("\nexample:", r.para_id)
    print("  raw  :", r.text_raw[:140].replace("\n", " "))
    print("  clean:", r.text_clean[:140].replace("\n", " "))

print("\n--- evidence alignment funnel (QASPER, 3,598 questions) ---")
print("questions with >=1 annotated evidence span :", int((qa.n_evidence > 0).sum()))
print("questions with >=1 span mapped to a para   :", int((qa.n_evidence_mapped > 0).sum()))
print("\nexcluded from the recall denominator, by reason:")
reason = qa.zero_gold_reason.replace("", "(has gold)").value_counts()
for k, v in reason.items():
    print(f"  {k:14s} {v:5d}")
print("\nanswerable:      ", int(qa.is_answerable.sum()))
print("float-only gold: ", int(qa.is_float_only.sum()))
print("eligible total:  ", int((qa.zero_gold_reason == "").sum()))

paragraphs whose raw text differs from cleaned text: 27457 (44.8%)

example: 1909.00694#p0001
  raw  : Affective events BIBREF0 are events that typically affect people in positive or negative ways. For example, getting money and playing sports
  clean: Affective events [REF] are events that typically affect people in positive or negative ways. For example, getting money and playing sports a

--- evidence alignment funnel (QASPER, 3,598 questions) ---
questions with >=1 annotated evidence span : 3167
questions with >=1 span mapped to a para   : 2991

excluded from the recall denominator, by reason:
  (has gold)      2991
  no_evidence      431
  float_only       174
  align_fail         2

answerable:       3267
float-only gold:  174
eligible total:   2991


## 4. Chunking: sentence packing with token budget and overlap

Algorithm (`src/rag_data/chunker.py`):

1. split each paragraph into sentences (pysbd);
2. greedily pack sentences until the **token budget** (512) is hit;
3. start the next chunk by backing up whole sentences worth of **overlap** (20% ≈ 102 tokens);
4. if a *single sentence* exceeds the budget, hard-split it at a token boundary and mark
   `is_truncated=True`;
5. section boundaries can be `hard` (never cross) or `soft` (allowed, recorded in `n_sections`).

The demo below uses the real BGE tokenizer on a real 888-token paragraph.

In [5]:
from tokenizers import Tokenizer

TOK = Tokenizer.from_file("data/raw/bge_tok/tokenizer.json")
SENT_RE = re.compile(r"(?<=[.!?])\s+")


def split_sentences(text):
    spans, start = [], 0
    for m in SENT_RE.finditer(text):
        if m.start() > start:
            spans.append((start, m.start()))
        start = m.end()
    if start < len(text):
        spans.append((start, len(text)))
    return spans or [(0, len(text))]


def pack_chunks(text, max_tokens=512, overlap=102):
    sents = split_sentences(text)
    out, i, n = [], 0, len(sents)
    while i < n:
        a = sents[i][0]
        if len(TOK.encode(text[a:sents[i][1]]).ids) > max_tokens:
            lo, hi = a, sents[i][1]
            while hi - lo > 1:
                mid = (lo + hi) // 2
                if len(TOK.encode(text[a:mid]).ids) <= max_tokens:
                    lo = mid
                else:
                    hi = mid
            b = lo
            out.append((a, b, len(TOK.encode(text[a:b]).ids), True))
            sents[i] = (b, sents[i][1])
            if b >= sents[i][1]:
                i += 1
            continue
        j = i
        while j + 1 < n and len(TOK.encode(text[a:sents[j + 1][1]]).ids) <= max_tokens:
            j += 1
        b = sents[j][1]
        out.append((a, b, len(TOK.encode(text[a:b]).ids), False))
        if j + 1 >= n:
            break
        k = j
        while k > i and len(TOK.encode(text[sents[k][0]:b]).ids) < overlap:
            k -= 1
        i = max(k, i + 1)
    return out


long_p = pa[pa.n_tokens > 512].iloc[0]
print(f"demo paragraph {long_p.para_id}: {long_p.n_tokens} tokens")
for a, b, t, trunc in pack_chunks(long_p.text_clean):
    print(f"  [{a:5d},{b:5d}) {t:3d} tok {'[hard-split]' if trunc else ''} | {long_p.text_clean[a:b][:52]}...")

print("\n--- chunk table statistics (QASPER default config) ---")
print("chunks:", len(ck), "| papers:", ck.paper_id.nunique(),
      f"| chunks/paper: {len(ck)/ck.paper_id.nunique():.2f}")
q = ck.n_tokens.quantile([0.5, 0.9, 0.95, 1.0]).astype(int)
print("n_tokens  p50/p90/p95/max:", list(q.values))
print("abstract chunks:", int(ck.is_abstract.sum()), f"({ck.is_abstract.mean()*100:.1f}%)")
print("hard-split chunks:", int(ck.is_truncated.sum()))

demo paragraph 1910.14497#p0025: 888 tokens
  [    0, 1548) 512 tok [hard-split] | 'accountant', 'acquaintance', 'actor', 'actress', 'a...
  [ 1548, 2690) 380 tok  | , 'marshal', 'mathematician', 'mechanic', 'midfielde...

--- chunk table statistics (QASPER default config) ---
chunks: 17653 | papers: 1169 | chunks/paper: 15.10
n_tokens  p50/p90/p95/max: [np.int64(495), np.int64(509), np.int64(511), np.int64(512)]
abstract chunks: 1176 (6.7%)
hard-split chunks: 29


## 5. Executable data contract

`src/rag_data/validate.py` turns the written contract into 19 machine checks across
structure, completeness, semantics and leakage. Any `FAIL` means the artifact must not be used.

This is the same validator used by `scripts/05_validate_contract.py`.

In [6]:
sys.path.insert(0, "src")
from rag_data import validate as V

rows_v = []
for r in ck.to_dict("records"):
    s = r["para_spans"]
    r["para_spans"] = [tuple(x) for x in json.loads(s)] if isinstance(s, str) else s
    rows_v.append(r)

para_len = dict(zip(pa.para_id, pa.n_chars))
para_text = dict(zip(pa.para_id, pa.text_clean))
qrows = [json.loads(line) for line in open(FILES["QASPER gold map"], encoding="utf-8")]

findings = []
findings += V.check_columns(rows_v, V.CHUNK_REQUIRED, V.FORBIDDEN_IN_PUBLIC)
findings += V.check_chunk_ids(rows_v)
findings += V.check_ordering(rows_v)
findings += V.check_neighbours(rows_v)
findings += V.check_token_budget(rows_v, 512)
findings += V.check_para_spans(rows_v, para_len)
findings += V.check_coverage(rows_v, para_len, para_text)
findings += V.check_no_gold_in_public(rows_v)
findings += V.check_gold(qrows, set(ck.chunk_id), set(pa.para_id))

for f in findings:
    print(f"  {f.status:4s} {f.check}")
print("\nsummary (pass, warn, fail):", V.summarise(findings))

  PASS columns.required
  PASS columns.extra
  PASS columns.leakage
  PASS id.unique
  PASS id.format
  PASS order.in_paper
  PASS order.in_section
  PASS graph.neighbours
  PASS size.within_budget
  PASS size.non_empty
  PASS spans.paired
  PASS spans.unique
  PASS spans.in_bounds
  PASS coverage.paragraphs
  PASS coverage.content
  PASS leakage.value_fields
  PASS gold.para_refs
  PASS gold.chunk_refs
  PASS gold.reachable

summary (pass, warn, fail): (19, 0, 0)


## 6. Dense retrieval

Production implementation (`baselines/member2_dense_rag/`):
encode chunks with `BAAI/bge-small-en-v1.5` → L2-normalise → FAISS `IndexFlatIP` →
cosine top-k, scoped to the question's paper for QASPER and globally for PubMedQA-graph.

Here the encoder is abstracted behind `encode()`. Without BGE installed it falls back to a
hashing bag-of-words encoder so the retrieval loop and metrics remain executable.

In [7]:
DIM = 4096
_encoder = None


def _tokens(s):
    return re.findall(r"[a-z0-9]+", s.lower())


def encode(texts):
    global _encoder
    if BACKEND == "bge" and _encoder is None:
        from sentence_transformers import SentenceTransformer
        _encoder = SentenceTransformer("BAAI/bge-small-en-v1.5")
    if BACKEND == "bge":
        return _encoder.encode(texts, normalize_embeddings=True, batch_size=32)
    m = np.zeros((len(texts), DIM), dtype=np.float32)
    for i, t in enumerate(texts):
        for w in _tokens(t):
            m[i, zlib.crc32(w.encode()) % DIM] += 1.0
        norm = np.linalg.norm(m[i])
        if norm:
            m[i] /= norm
    return m


t0 = time.time()
C = encode(g_ck.text.tolist())
print("chunk matrix:", C.shape, f"| encoded in {time.time()-t0:.1f}s | backend={BACKEND}")

chunk matrix: (3707, 4096) | encoded in 0.1s | backend=hashing


In [8]:
qid = "21645374"
qtext = g_qa[g_qa.question_id == qid].iloc[0].question
qv = encode([qtext])[0]
scores = C @ qv
top = np.argsort(-scores)[:5]

print("query:", qtext)
print("gold :", list(g_qa[g_qa.question_id == qid].iloc[0].gold_chunk_ids))
print()
for rank, idx in enumerate(top, 1):
    r = g_ck.iloc[idx]
    print(f"#{rank}  {r.chunk_id}  |  {r.section_name:<12s} | score {scores[idx]:.3f}")
    print(f"     {r.text[:78]}...")

query: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
gold : ['W2018063873#s00#c000', 'W2018063873#s01#c000']

#1  W2082571460#s00#c000  |  PURPOSE      | score 0.324
     A possible role for fondaparinux as a bridging agent in the perioperative sett...
#2  W2018063873#s00#c000  |  BACKGROUND   | score 0.322
     Programmed cell death (PCD) is the regulated death of cells within an organism...
#3  W2018063873#s01#c000  |  RESULTS      | score 0.274
     The following paper elucidates the role of mitochondrial dynamics during devel...
#4  W2094548557#s04#c000  |  MAIN OUTCOME MEASURE | score 0.267
     Death....
#5  W2262039363#s00#c000  |  BACKGROUND   | score 0.260
     Anchoring vignettes are brief texts describing a hypothetical character who il...


## 7. Evaluation

For PubMedQA-graph the retrieval pool is the **entire corpus** (no paper scoping) and gold is
the set of chunks belonging to the source paper:

$$\mathrm{Hit@}k=\frac{1}{N}\sum_i \mathbb{1}\left[\text{top-}k_i \cap \text{gold}_i \neq \varnothing\right],
\qquad
\mathrm{MRR}=\frac{1}{N}\sum_i \frac{1}{\text{rank}_i}$$

For QASPER the metric is **paragraph-level macro recall** over the 890 dev questions with gold —
never chunk-level (chunk ids change with the config) and never with no-gold questions scored as zero.

In [9]:
gold = {r.question_id: set(r.gold_chunk_ids) for r in g_qa.itertuples()}
chunk_ids = g_ck.chunk_id.tolist()
id2idx = {c: i for i, c in enumerate(chunk_ids)}
qids = list(gold)

Q = encode(g_qa.question.fillna("").tolist())
S = Q @ C.T
print("score matrix:", S.shape)


def hit_at(k, ranking):
    return float(np.mean([bool(set(ranking[i][:k]) & gold[q]) for i, q in enumerate(qids)]))


def mrr_at(k, ranking):
    vals = []
    for i, q in enumerate(qids):
        rr = 0.0
        for pos, c in enumerate(ranking[i][:k], 1):
            if c in gold[q]:
                rr = 1.0 / pos
                break
        vals.append(rr)
    return float(np.mean(vals))


order = np.argsort(-S, axis=1)[:, :10]
rk = [[chunk_ids[j] for j in row] for row in order]

print(f"questions: {len(qids)} | backend: {BACKEND}")
print(f"Hit@1 {hit_at(1, rk):.4f} | Hit@3 {hit_at(3, rk):.4f} | Hit@10 {hit_at(10, rk):.4f} | MRR {mrr_at(10, rk):.4f}")
print("\nNote: the hashing backend rewards lexical overlap (questions are derived from abstracts),")
print("so these numbers are inflated relative to BGE. Use them as a plumbing baseline only.")

score matrix: (951, 3707)
questions: 951 | backend: hashing
Hit@1 0.6856 | Hit@3 0.7771 | Hit@10 0.8475 | MRR 0.7373

Note: the hashing backend rewards lexical overlap (questions are derived from abstracts),
so these numbers are inflated relative to BGE. Use them as a plumbing baseline only.


## 8. From PubMedQA to a citation graph (scripts/20 → 21)

The original PubMedQA has **no citations**. The expansion pipeline:

| stage | what happens | outcome |
|---|---|---|
| P0 | resolve 1,000 PMIDs on OpenAlex | 951 resolved (49 not indexed) |
| P1 | collect `referenced_works` per paper | 24,457 directed citation edges |
| P2 | expand the corpus with references | 23,792 unique references |
| P2' | keep only references cited ≥ 2 times | 444 “bridge” papers (the coupling core) |
| P3 | chunk the corpus (CONTEXTS / abstracts) | 1,395 papers → 3,707 chunks |
| P4 | build edges | 1,042 `cites` + 2,312 `adjacency` |
| P5 | redefine QA | gold = source paper's chunks, 951 questions |

**Why not keep all 23,792 references?** The 1-hop citation graph is a near-star: only 448 references
are cited more than once. The other ~23k are dead ends that add 3× corpus size, 41% empty-text nodes,
and zero cross-document structure. Keeping the coupling core makes the graph connected where it matters.

In [10]:
print("--- corpus composition ---")
print(g_pa.role.value_counts().to_string())
print("\nchunks per role:")
print(g_ck.merge(g_pa[["paper_id", "role"]], on="paper_id").role.value_counts().to_string())
print("\nabstract coverage by role (%):")
print((g_pa.groupby("role").has_abstract.mean() * 100).round(1).to_string())

print("\n--- graph edges ---")
print(g_ed.edge_type.value_counts().to_string())
cites = g_ed[g_ed.edge_type == "cites"]
deg_out = cites.groupby("src_id").size()
deg_in = cites.groupby("dst_id").size()
print(f"\ncites: {len(cites)} edges | papers with >=1 outgoing: {deg_out.size} | >=1 incoming: {deg_in.size}")
print("top cited bridges:")
for pid, d in deg_in.sort_values(ascending=False).head(5).items():
    title = g_pa.loc[g_pa.paper_id == pid, "title"].iloc[0]
    print(f"  {pid}  cited by {d:2d}  | {title[:60]}")

--- corpus composition ---
role
source       951
reference    444

chunks per role:
role
source       3192
reference     515

abstract coverage by role (%):
role
reference     67.8
source       100.0

--- graph edges ---
edge_type
adjacency    2312
cites        1042

cites: 1042 edges | papers with >=1 outgoing: 425 | >=1 incoming: 451
top cited bridges:
  W3147894994  cited by 10  | Regression Models and Life-Tables
  W1979300931  cited by 10  | Nonparametric Estimation from Incomplete Observations
  W2159011576  cited by  9  | Diagnostic and Statistical Manual of Mental Disorders
  W2166281097  cited by  9  | The Hospital Anxiety and Depression Scale
  W2164777277  cited by  8  | The Measurement of Observer Agreement for Categorical Data


In [11]:
pid = "W2147896631"
print("paper:", pid, "|", g_pa.loc[g_pa.paper_id == pid, "title"].iloc[0])
print("questions about it:", g_qa[g_qa.source_paper_id == pid].question.tolist())
print("\ncitation neighbours inside the corpus:")
for x in g_pa[g_pa.paper_id.isin(cites[cites.src_id == pid].dst_id)].head(6).itertuples():
    print(f"  -> {x.paper_id} | {x.title[:64]}")

pid2 = "W2018063873"
print("\ncounter-example (no bridge):", pid2, "|", g_pa.loc[g_pa.paper_id == pid2, "title"].iloc[0])
print("  outgoing cites in corpus:", int((cites.src_id == pid2).sum()),
      "| incoming:", int((cites.dst_id == pid2).sum()))

paper: W2147896631 | Is Nasogastric Decompression Useful in Prevention of Leaks After Laparoscopic Sleeve Gastrectomy? A Randomized Trial
questions about it: ['Is nasogastric decompression useful in prevention of leaks after laparoscopic sleeve gastrectomy?']

citation neighbours inside the corpus:
  -> W1927710737 | Obesity as a medical problem
  -> W1996699012 | Laparoscopic Sleeve Gastrectomy—Volume and Pressure Assessment
  -> W1993208462 | Gastric Leak After Laparoscopic-Sleeve Gastrectomy for Obesity
  -> W1985794584 | Reinforcing the Staple Line During Laparoscopic Sleeve Gastrecto
  -> W2022282990 | Long-term Results of Laparoscopic Sleeve Gastrectomy for Obesity
  -> W2028015032 | Reinforcement Does Not Necessarily Reduce the Rate of Staple Lin

counter-example (no bridge): W2018063873 | Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
  outgoing cites in corpus: 0 | incoming: 0


## 9. Graph-enhanced retrieval (a minimal CG-RAG realisation)

CG-RAG's LeSeGR is a trained Graph Transformer. Before paying for that, validate the mechanism
with a cheap, training-free version:

1. dense top-10 → **seed chunks**;
2. seed chunk → paper → **citation neighbours** (1 hop);
3. neighbour chunks enter the candidate pool;
4. rescore: $\text{score}(q,c)=\text{sim}(q,c)+\lambda\cdot\max_{s\in\text{seeds}}\text{sim}(c,s)$.

If 1-hop expansion does not help, a trained graph encoder is unlikely to rescue the setup —
that is the “cheap falsification first” order recommended in the design review.

In [12]:
paper_of = dict(zip(g_ck.chunk_id, g_ck.paper_id))
chunks_of = {}
for r in g_ck.itertuples():
    chunks_of.setdefault(r.paper_id, []).append(r.chunk_id)

nbr = {}
for r in cites.itertuples():
    nbr.setdefault(r.src_id, set()).add(r.dst_id)
    nbr.setdefault(r.dst_id, set()).add(r.src_id)

LAM = 0.5
BASE_K = 50


def expand_rank(i, topk=10, base_k=BASE_K):
    seed = [chunk_ids[j] for j in np.argsort(-S[i])[:topk]]
    pool = [chunk_ids[j] for j in np.argsort(-S[i])[:base_k]]
    papers = {paper_of[c] for c in seed}
    cand_papers = set(papers)
    for p in papers:
        cand_papers |= nbr.get(p, set())
    cand = {c for p in cand_papers for c in chunks_of.get(p, [])} | set(pool)
    ci = np.array([id2idx[c] for c in cand])
    base = S[i, ci]
    si = np.array([id2idx[c] for c in seed])
    neigh = (C[ci] @ C[si].T).max(axis=1)
    total = base + LAM * neigh
    order = ci[np.argsort(-total)]
    return [chunk_ids[j] for j in order[:topk]], len(cand)


t0 = time.time()
rk2, sizes = [], []
for i in range(len(qids)):
    r, n = expand_rank(i)
    rk2.append(r)
    sizes.append(n)

changed = sum(1 for a, b in zip(rk, rk2) if a != b)
recovered = sum(
    1 for i, q in enumerate(qids)
    if not (set(rk[i][:10]) & gold[q]) and (set(rk2[i][:10]) & gold[q])
)
print(f"expansion: {time.time()-t0:.1f}s | mean candidates {np.mean(sizes):.0f} (dense pool {BASE_K})")
print(f"questions whose top-10 changed: {changed} | newly recovered gold: {recovered}")
print()
print("          Hit@1    Hit@3    Hit@10     MRR")
print(f"dense   {hit_at(1,rk):7.4f} {hit_at(3,rk):8.4f} {hit_at(10,rk):9.4f} {mrr_at(10,rk):7.4f}")
print(f"1-hop   {hit_at(1,rk2):7.4f} {hit_at(3,rk2):8.4f} {hit_at(10,rk2):9.4f} {mrr_at(10,rk2):7.4f}")

expansion: 1.6s | mean candidates 92 (dense pool 50)
questions whose top-10 changed: 92 | newly recovered gold: 0

          Hit@1    Hit@3    Hit@10     MRR
dense    0.6856   0.7771    0.8475  0.7373
1-hop    0.6856   0.7771    0.8475  0.7375


## 10. QASPER within-paper baseline and the oracle ceiling

QASPER questions are paper-bound, so retrieval is scoped to the target paper. `oracle_recall@k`
is the model-free ceiling imposed purely by chunking: rank the paper's chunks perfectly and see
how much gold evidence is reachable within *k* chunks. Any method exceeding it is buggy.

The baseline below is Member 2's BGE + FAISS run, read from its committed metrics.

In [13]:
# oracle ceiling on dev (same para-level semantics as the reported metric)
dev = qa[(qa.split == "dev") & (qa.zero_gold_reason == "")].copy()
paper_chunks = {}
for r in ck.itertuples():
    paper_chunks.setdefault(r.paper_id, []).append(set(r.para_ids))

KS = (1, 3, 5, 10)
acc = {k: 0.0 for k in KS}
n = 0
for r in dev.itertuples():
    G = set(r.gold_para_ids)
    if not G:
        continue
    cands = paper_chunks.get(r.paper_id, [])
    chosen, covered, step = set(), set(), {}
    for s in range(1, max(KS) + 1):
        best, best_new = None, -1
        for i, c in enumerate(cands):
            if i in chosen:
                continue
            new = len((covered | c) & G)
            if new > best_new:
                best_new, best = new, i
        if best is None:
            break
        chosen.add(best)
        covered |= cands[best]
        step[s] = len(covered & G)
        if len(covered & G) == len(G):
            break
    run = 0
    for k in KS:
        for s in range(1, k + 1):
            run = max(run, step.get(s, 0))
        acc[k] += run / len(G)
    n += 1

print(f"dev questions with gold: {n}")
print("                    k=1     k=3     k=5    k=10")
print("oracle recall  " + "  ".join(f"{acc[k]/n*100:6.2f}" for k in KS))

p = Path("baselines/member2_dense_rag/docs/qasper_dev_metrics.json")
if p.exists():
    d = json.loads(p.read_text(encoding="utf-8"))
    print(f"\nM2 dense baseline | split={d['split']} | total={d['total_questions']} | eligible={d['eligible_questions']}")
    print("                Recall@k     MRR")
    for row in d["metrics"]:
        print(f"  k={row['k']:<2d}          {row['macro_paragraph_recall']*100:6.2f}%   {row['mrr']:.4f}")
else:
    print("\nM2 metrics file not found")

dev questions with gold: 890
                    k=1     k=3     k=5    k=10
oracle recall   88.42   99.76   99.99  100.00

M2 dense baseline | split=dev | total=1005 | eligible=890
                Recall@k     MRR
  k=1            33.70%   0.4213
  k=3            63.00%   0.5569
  k=5            77.33%   0.5836
  k=10           94.37%   0.6006


## 11. Findings and honest limitations

**What the numbers say**

1. Dense retrieval on QASPER is **precision-bound, not coverage-bound**: recall@10 is 94.37% while
   recall@1 is 33.70%, against an oracle ceiling of 88.42% at k=1. The evidence is almost always
   in the top-10; ranking it to the top is the bottleneck → rerankers/hybrid retrieval are the next lever.
2. On PubMedQA-graph, 1-hop citation expansion changed 92 question rankings but recovered **zero**
   additional gold chunks under the cheap encoder. The graph mechanism ran; it simply had nothing
   to add on that backend. A trained graph encoder should only be attempted after this is re-checked
   with real BGE embeddings.
3. Only **423 / 951 (44%)** of source papers have any cross-document citation edge. For the other 56%,
   the graph can only contribute intra-document structure. Report grouped results, or the gain is diluted.

**What is out of scope**

* PubMedQA-graph is **abstract-level**: one paper ≈ 2.7 chunks, so this is closer to paper retrieval
  than paragraph retrieval. Full-text (PMC) would change the granularity.
* No LLM generation: answers (yes/no/maybe) require a reader; the retrieval half is what is measured here.
* The paper's 7,849-paper graph is **not reproducible**: its reference-extraction policy is unpublished.
  Compare within our own pool (dense vs graph), never against its Hit@1 = 0.961.

**Two denominators that must never be mixed up**

* QASPER: 2,991 eligible overall, **890 on dev** — not 3,598.
* PubMedQA-graph: **951** — not 1,000 (49 PMIDs are absent from OpenAlex).

In [14]:
print("SUMMARY")
print("=" * 62)
print(f"{'item':<34}{'value':>26}")
print("-" * 62)
vals = [
    ("QASPER questions / eligible", f"{len(qa)} / {int((qa.zero_gold_reason=='').sum())}"),
    ("QASPER chunks", f"{len(ck)}"),
    ("QASPER dense recall@1 / @10", "33.70% / 94.37%"),
    ("QASPER oracle @1 / @5", f"{acc[1]/n*100:.2f}% / {acc[5]/n*100:.2f}%"),
    ("PubMedQA-graph papers / chunks", f"{g_pa.paper_id.nunique()} / {len(g_ck)}"),
    ("PubMedQA-graph cites / adjacency", f"{int((g_ed.edge_type=='cites').sum())} / {int((g_ed.edge_type=='adjacency').sum())}"),
    ("PubMedQA-graph questions", f"{len(qids)}"),
    ("dense Hit@1 / Hit@10 / MRR", f"{hit_at(1,rk):.4f} / {hit_at(10,rk):.4f} / {mrr_at(10,rk):.4f}"),
    ("1-hop new gold recovered", f"{recovered}"),
    ("active backend", BACKEND),
]
for k, v in vals:
    print(f"{k:<34}{v:>26}")
print("=" * 62)

SUMMARY
item                                                   value
--------------------------------------------------------------
QASPER questions / eligible                      3598 / 2991
QASPER chunks                                          17653
QASPER dense recall@1 / @10                  33.70% / 94.37%
QASPER oracle @1 / @5                        88.42% / 99.99%
PubMedQA-graph papers / chunks                   1395 / 3707
PubMedQA-graph cites / adjacency                 1042 / 2312
PubMedQA-graph questions                                 951
dense Hit@1 / Hit@10 / MRR          0.6856 / 0.8475 / 0.7373
1-hop new gold recovered                                   0
active backend                                       hashing


## 12. Reproducing the full pipeline

```bash
# data (already committed; these rebuild it from raw QASPER)
python scripts/00_profile_dataset.py --data-dir data/raw/qasper --out-dir docs
python scripts/01_prepare_corpus.py  --data-dir data/raw/qasper --processed-dir data/processed
python scripts/02_align_evidence.py  --data-dir data/raw/qasper --processed-dir data/processed
python scripts/03_build_chunks.py    --chunk-size 512 --overlap 0.2 --section-boundary soft
python scripts/05_validate_contract.py

# PubMedQA citation graph
python scripts/20_expand_pubmedqa_corpus.py --min-cited 2
python scripts/21_build_pubmedqa_graph.py

# tests
python -m pytest tests -q
```

| what | where |
|---|---|
| contract (authoritative) | `docs/contract/data_contract.md` |
| chunker | `src/rag_data/chunker.py` |
| validator | `src/rag_data/validate.py` |
| dense baseline (BGE + FAISS) | `baselines/member2_dense_rag/` |
| citation-graph pipeline | `scripts/20_*.py`, `scripts/21_*.py` |
| graph experiments (GNN) | `data_process/`, `data_process/gnn_runs/` |
| interactive demo | `demo/` |